## 01 · Extract data

Reads five of the raw files in `data/raw/`, keeps only the series the project needs, standardises the dates, and saves one CSV per source to `data/processed/`. The USGS production data is used later, for the index weights.

| Source | File | Series |
|---|---|---|
| World Bank Pink Sheet | `worldbank_pinksheet_monthly.xlsx` | metal prices, Brent oil, natural gas |
| New York Fed | `nyfed_gscpi.xls` | supply chain pressure (GSCPI) |
| CPB World Trade Monitor | `cpb_world_trade_monitor_2026m07.xlsx` | industrial production |
| BIS via FRED | `bis_usd_broad_neer_fred.xlsx` | US dollar index |
| IMF | `imf_pcps_metals_monthly.csv` | cobalt, lithium |

---

*Code written with help of Claude (Anthropic), asked to produce code that extracts the needed series from my raw data files and summarises their coverage.*

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

def to_month_start(dates):
    """Convert dates such as '2000M01', '2000m01' or 2000-01-31 to the first day of the month."""
    text = pd.Series(dates).astype(str).str.replace("M", "-").str.replace("m", "-")
    return pd.to_datetime(text, errors="coerce").dt.to_period("M").dt.to_timestamp().values

In [2]:
# WorldBank prices database
# Monthly prices for the four core metals, zinc and lead (comparison group), Brent oil and the natural gas index. Source: World Bank Commodity Price Data (CC BY 4.0).

PINK_columns = {
    "Copper": "copper", "Aluminum": "aluminium", "Nickel": "nickel", "Tin": "tin",
    "Zinc": "zinc", "Lead": "lead",
    "Crude oil, Brent": "brent", "Natural gas index": "naturalgas_index",
}

pinksheet_raw_data = pd.read_excel(RAW / "worldbank_pinksheet_monthly.xlsx", sheet_name="Monthly Prices", skiprows=4)
pinksheet_raw_data = pinksheet_raw_data.iloc[1:]                                   

prices_worldbank = pinksheet_raw_data[list(PINK_columns)].rename(columns=PINK_columns)
prices_worldbank = prices_worldbank.apply(pd.to_numeric, errors="coerce")  
prices_worldbank.insert(0, "date", to_month_start(pinksheet_raw_data.iloc[:, 0]))

In [3]:
# New York Fed GSCPI data 
gscpi = pd.read_excel(RAW / "nyfed_gscpi.xls", sheet_name="GSCPI Monthly Data",
                      skiprows=5, header=None, usecols=[0, 1], names=["date", "gscpi"])
gscpi["date"] = to_month_start(pd.to_datetime(gscpi["date"], format="%d-%b-%Y"))

In [4]:
# CPB world trade monitor
# World industrial production (excluding construction, 2021 = 100, seasonally adjusted), the demand control. The file lists each series as a row with months across the columns, so the three series are selected by their CPB code and the table is flipped. Source: CPB Netherlands Bureau for Economic Policy Analysis.

CPB_codes = {"ipz_w1_qnmi_sp": "ind_production_world", "ipz_w1_qnmi_sm": "ind_production_world_importweighted", "ipz_cn_qnmi_sp": "ind_production_china"}

cpb = pd.read_excel(RAW / "cpb_world_trade_monitor_2026m07.xlsx", sheet_name="inpro_out", header=None)
date_cols = cpb.columns[cpb.iloc[3].astype(str).str.match(r"\d{4}m\d{2}")]   # 

industrial_production = cpb.set_index(2).loc[list(CPB_codes), date_cols].T   
industrial_production.columns = [CPB_codes[c] for c in industrial_production.columns]
industrial_production = industrial_production.astype(float).reset_index(drop=True)
industrial_production.insert(0, "date", to_month_start(cpb.loc[3, date_cols]))

In [5]:
# US dollar index (BIS, via Fred)
# The US dollar's value against a broad basket of currencies. 'NEER' stands for nominal effective exchange rate. Higher = stronger US dollar. 

bis_raw_data = pd.read_excel(RAW / "bis_usd_broad_neer_fred.xlsx", sheet_name="Monthly", header=None)
start = bis_raw_data.index[bis_raw_data[0] == "observation_date"][0] + 1    

usd = pd.DataFrame({
    "date": to_month_start(bis_raw_data.iloc[start:, 0]),
    "us_dollar_neer": pd.to_numeric(bis_raw_data.iloc[start:, 1]).values,
})

In [6]:
# IMF Primary Commodity Prices
# Cobalt and lithium (99%, battery grade), plus copper, aluminium, nickel and tin for a consistency check against the World Bank prices. Lithium starts in June 2012. The IMF labels cobalt in US dollars per pound, but the values are per tonne. Source: International Monetary Fund, Primary Commodity Prices.

IMF_codenames = {
    "G001.PCOBA.USD.M": "cobalt",
    "G001.PLITH.USD.M": "lithium",
    "G001.PCOPP.USD.M": "copper_imf",
    "G001.PALUM.USD.M": "aluminium_imf",
    "G001.PNICK.USD.M": "nickel_imf",
    "G001.PTIN.USD.M":  "tin_imf",
}

imf_raw_data = pd.read_csv(RAW / "imf_pcps_metals_monthly.csv")
imf_raw_data = imf_raw_data.set_index("SERIES_CODE").loc[list(IMF_codenames)]   

date_columns = [c for c in imf_raw_data.columns if c[:4].isdigit()]      
prices_imf = imf_raw_data[date_columns].T                                         
prices_imf.columns = [IMF_codenames[c] for c in prices_imf.columns]                  

prices_imf.index = pd.to_datetime(prices_imf.index.str.replace("-M", "-") + "-01")   
prices_imf = prices_imf.apply(pd.to_numeric, errors="coerce").rename_axis("date").reset_index()

In [7]:
# save data as .csv files 
outputs = {"prices_worldbank.csv": prices_worldbank, "gscpi.csv": gscpi,
           "industrial_production.csv": industrial_production, "usd.csv": usd,
           "prices_imf.csv": prices_imf}
for file, df in outputs.items():
    df.to_csv(PROCESSED / file, index=False)
    print(f"saved {file:28s} {len(df):5d} rows")

saved prices_worldbank.csv           800 rows
saved gscpi.csv                      344 rows
saved industrial_production.csv      319 rows
saved usd.csv                        391 rows
saved prices_imf.csv                 333 rows


In [8]:
# Data coverage: short descriptive analysis

rows = []
for file, table in outputs.items():
    for col in table.columns[1:]:                       
        available = table.dropna(subset=[col])
        expected = pd.date_range(available["date"].min(), available["date"].max(), freq="MS")
        rows.append([file, col,
                     available["date"].min().strftime("%Y-%m"),
                     available["date"].max().strftime("%Y-%m"),
                     len(available),
                     len(expected) - len(available)])

coverage = pd.DataFrame(rows, columns=["file", "series", "first_date", "last_date", "number_of_rows", "NA"])
coverage

,file,series,first_date,last_date,number_of_rows,NA
0,prices_worldbank.csv,copper,1960-01,2026-08,800,0
1,prices_worldbank.csv,aluminium,1960-01,2026-08,800,0
2,prices_worldbank.csv,nickel,1960-01,2026-08,800,0
3,prices_worldbank.csv,tin,1960-01,2026-08,800,0
4,prices_worldbank.csv,zinc,1960-01,2026-08,800,0
5,prices_worldbank.csv,lead,1960-01,2026-08,800,0
6,prices_worldbank.csv,brent,1960-01,2026-08,800,0
7,prices_worldbank.csv,naturalgas_index,1977-01,2026-08,596,0
8,gscpi.csv,gscpi,1998-01,2026-08,344,0
9,industrial_production.csv,ind_production_world,2000-01,2026-07,319,0


In [9]:
# Do the IMF and the World Bank agree?
# Correlation of monthly price changes for the four metals that appear in both sources. Values close to 1 mean the two sources report the same prices, so IMF cobalt and lithium can be used alongside World Bank prices.

both_sources = prices_worldbank.merge(prices_imf, on="date")
for metal in ["copper", "aluminium", "nickel", "tin"]:
    worldbank_change = np.log(both_sources[metal]).diff()
    imf_change = np.log(both_sources[metal + "_imf"]).diff()
    print(f"{metal:10s} correlation of monthly changes: {worldbank_change.corr(imf_change):.3f}")

copper     correlation of monthly changes: 0.999
aluminium  correlation of monthly changes: 0.999
nickel     correlation of monthly changes: 0.999
tin        correlation of monthly changes: 0.999
